In [7]:
# 1. Install required packages
!pip install fastapi uvicorn transformers pyngrok nest-asyncio

In [8]:
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import pipeline
import uvicorn
from threading import Thread

# Allow nested event loops in Colab
nest_asyncio.apply()

app = FastAPI()

# Load a public model that doesn't require a login (e.g., GPT-2)
print("Loading model...")
generator = pipeline('text-generation', model='gpt2')
print("Model loaded successfully!")

class PromptRequest(BaseModel):
    prompt: str
    max_length: int = 50

@app.post("/generate")
def generate_text(request: PromptRequest):
    res = generator(request.prompt, max_length=request.max_length, num_return_sequences=1)
    return {"generated_text": res[0]['generated_text']}

# Function to run the server in a background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = Thread(target=run_server)
server_thread.start()


Loading model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded successfully!


In [11]:
from pyngrok import ngrok
from google.colab import userdata

NGROK_TOKEN = userdata.get('NGROK_AUTH')
ngrok.set_auth_token(NGROK_TOKEN)

# Open a HTTP tunnel on port 8000
public_url = ngrok.connect(8000)
print(f"🚀 Your Public API URL is: {public_url.public_url}")


🚀 Your Public API URL is: https://reentry-subject-mannish.ngrok-free.dev


In [18]:
import requests

api_url = "https://reentry-subject-mannish.ngrok-free.dev/generate"

payload = {
    "prompt": "Deep learning is changing the world by",
    "max_length": 60
}

response = requests.post(api_url, json=payload)
print(response.json())
res = response.json()
print(res['generated_text'])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=60) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     34.16.209.206:0 - "POST /generate HTTP/1.1" 200 OK
{'generated_text': 'Deep learning is changing the world by allowing scientists to make predictions that are more accurate than previous predictions.\n\n"We know about a lot of things, and we\'re not looking to predict everything," says co-author Dr. H.C. Hodge, a professor of psychology at the University of Pennsylvania. "We\'re interested in the things that could be more dramatic."\n\nThe research was funded by the National Institutes of Health.\n\nThe research was published online May 2 in the journal Science.\n\nThe researchers used a new kind of machine learning algorithm that is able to model the neural network of the brain, and has shown it can predict what\'s likely to be real.\n\nCitation: Hodge, Hodge, V.A., and W.V. van der Molen, "Decoding human brain networks," Science, May 2, 2014. doi:10.1126/science.aac402711'}
Deep learning is changing the world by allowing scientists to make predictions that are more accurate